In [1]:
%pip -q install gymnasium numpy

In [2]:
import gymnasium as gym
import numpy as np

In [3]:
# Create the FrozenLake environment
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)

# Access the underlying environment to read its transition model
env_model = env.unwrapped

# Number of states and actions
n_states = env.observation_space.n
n_actions = env.action_space.n

print("Number of states:", n_states)
print("Number of actions:", n_actions)
print("Actions: 0=Left, 1=Down, 2=Right, 3=Up")

Number of states: 16
Number of actions: 4
Actions: 0=Left, 1=Down, 2=Right, 3=Up


In [4]:
# Parameters
gamma = 0.99
theta = 1e-8

# Fixed random policy: every action has equal probability
policy = np.ones((n_states, n_actions)) / n_actions

# Initial value estimate
V = np.zeros(n_states)

print("Discount factor (gamma):", gamma)
print("Convergence threshold (theta):", theta)
print("Policy action probabilities:", policy[0])

Discount factor (gamma): 0.99
Convergence threshold (theta): 1e-08
Policy action probabilities: [0.25 0.25 0.25 0.25]


In [5]:
def policy_evaluation(env, policy, gamma=0.99, theta=1e-8):
    """Estimate V_pi(s) for a fixed policy using iterative policy evaluation.

    Args:
        env: Unwrapped Gymnasium FrozenLake environment with env.P transition model.
        policy: Array shaped (number_of_states, number_of_actions).
        gamma: Discount factor.
        theta: Convergence threshold.

    Returns:
        V: Converged state-value estimates.
        iteration: Number of sweeps used to converge.
    """
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    V = np.zeros(n_states, dtype=float)
    iteration = 0

    while True:
        delta = 0.0
        iteration += 1

        for state in range(n_states):
            old_value = V[state]
            new_value = 0.0

            for action in range(n_actions):
                action_probability = policy[state, action]

                for probability, next_state, reward, terminated in env.P[state][action]:
                    # A terminal transition has no future-state value.
                    future_value = 0.0 if terminated else V[next_state]
                    new_value += action_probability * probability * (
                        reward + gamma * future_value
                    )

            V[state] = new_value
            delta = max(delta, abs(old_value - new_value))

        if delta < theta:
            break

    return V, iteration

V, iterations = policy_evaluation(env_model, policy, gamma=gamma, theta=theta)

In [7]:
# Display the results
print("Name: S Lalit Chandran")
print("Register Number: 212223240077")
print("Number of Iterations:", iterations)

print("\\nState-Value Function (all 16 states):")
print(np.round(V, 6))

print("\\nState-Value Function as a 4x4 Grid:")
print(np.round(V.reshape(4, 4), 4))

print("\\nState-value grid interpretation:")
print("S = Start, F = Frozen safe tile, H = Hole, G = Goal")
desc = env_model.desc.astype("U1")
for row in range(4):
    row_text = []
    for col in range(4):
        state = row * 4 + col
        row_text.append(f"{desc[row, col]}:{V[state]:.4f}")
    print("  ".join(row_text))

env.close()

Name: S Lalit Chandran
Register Number: 212223240077
Number of Iterations: 54
\nState-Value Function (all 16 states):
[0.012356 0.010424 0.019338 0.009478 0.014787 0.       0.038894 0.
 0.032602 0.084338 0.137811 0.       0.       0.170345 0.433579 0.      ]
\nState-Value Function as a 4x4 Grid:
[[0.0124 0.0104 0.0193 0.0095]
 [0.0148 0.     0.0389 0.    ]
 [0.0326 0.0843 0.1378 0.    ]
 [0.     0.1703 0.4336 0.    ]]
\nState-value grid interpretation:
S = Start, F = Frozen safe tile, H = Hole, G = Goal
S:0.0124  F:0.0104  F:0.0193  F:0.0095
F:0.0148  H:0.0000  F:0.0389  H:0.0000
F:0.0326  F:0.0843  F:0.1378  H:0.0000
H:0.0000  F:0.1703  F:0.4336  G:0.0000
